# Part 5 · Agent Substrate: your agents in a gVisor sandbox

Parts 1-4 secured *service* and *agent* traffic. This part is about **where an agent runs**. Solo's kagent **Agent Substrate** runs each agent as an **actor inside a gVisor sandbox** on a pool of pre-warmed workers, with memory snapshots for fast resume, you deploy a `SandboxAgent` instead of an ordinary pod.

> **Beta / off by default.** Substrate ships in kagent-enterprise ≥ v0.5.2 and is disabled until you turn it on. It runs on kind with **gVisor (runsc)**, no `/dev/kvm` needed, because gVisor is a userspace kernel.

### The substrate components

`substrate-up.sh` turns on kagent's **Agent Substrate** engine (API group `ate.dev`; every component is prefixed `ate*`). What it adds to the `kagent` namespace:

| Component | Role |
|---|---|
| **SandboxAgent** (CRD) | the agent you deploy, runs as a gVisor actor, not an ordinary pod |
| **WorkerPool** (`ate.dev`) | pool of pre-warmed gVisor workers (`ateom-gvisor` image) |
| **ate-api-server** | control-plane gRPC API (`kagent-api.kagent.svc:443`), manages actors, resolves their env |
| **ate-controller** | reconciles `ActorTemplate`s / golden actors |
| **atelet** (DaemonSet) | downloads `runsc` and runs actors under gVisor on each node |
| **atenet-router** | actor networking |
| **valkey** | actor / worker state store |


## Connect · run this first

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
# clear strays (dev servers, other labs' stale port-forwards) off this demo's local ports;
# Docker publishes and this suite's own kubectl forwards survive
./demo-scripts/free-ports.sh 18083
export CTX KAGENT_NS
export KENT_CRDS_CHART="oci://us-docker.pkg.dev/solo-public/kagent-enterprise-helm/charts/kagent-enterprise-crds"
export KENT_CHART="oci://us-docker.pkg.dev/solo-public/kagent-enterprise-helm/charts/kagent-enterprise"
export KAGENT_ENT_VERSION="${KAGENT_ENT_VERSION:-0.5.2}"
echo "context: $CTX ; kagent-enterprise target: $KAGENT_ENT_VERSION"
kubectl --context $CTX version -o json 2>/dev/null | python3 -c "import sys,json;print('k8s:',json.load(sys.stdin)['serverVersion']['gitVersion'],'(substrate needs >=1.33)')" 2>/dev/null || true

## How substrate gets enabled

Substrate is **off by default**. The **Enable substrate** cell below turns it on: it installs the substrate control plane and a gVisor `WorkerPool`, a pool of pre-warmed, sandboxed workers that your agents run on. It's idempotent, so it's safe to re-run. On Apple Silicon it uses the arm64 `ateom-gvisor` worker image.

In [ ]:
: "${CTX:=kind-substrate}"
# ONE-TIME: enable Agent Substrate, installs the substrate control plane + a gVisor
# WorkerPool. Idempotent (already done if you ran setup with ENABLE_SUBSTRATE=true). ~several min.
bash "$(git rev-parse --show-toplevel)/vision-demo-2026/demo-scripts/substrate-cluster.sh"

## 5.1 · Step 1: prove the agent runs in a gVisor sandbox

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 260" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="260" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">Step 1 · Where your agent actually runs: a gVisor-sandboxed actor</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="v" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#8b5cf6"/></marker></defs><rect x="14" y="66" width="116" height="46" rx="8" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.5"/><text x="72" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#312e81">SandboxAgent</text><text x="72" y="101" text-anchor="middle" font-size="8" fill="#4338ca">(kagent CRD)</text><line x1="130" y1="89" x2="168" y2="89" stroke="#334155" stroke-width="1.7" marker-end="url(#n)"/><rect x="170" y="66" width="120" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.4"/><text x="230" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">kagent-</text><text x="230" y="101" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">controller</text><line x1="290" y1="89" x2="330" y2="89" stroke="#334155" stroke-width="1.7" marker-end="url(#n)"/><text x="310" y="80" text-anchor="middle" font-size="7.5" fill="#475569">ActorTemplate</text><rect x="332" y="66" width="110" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.4"/><text x="387" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">ate-api-</text><text x="387" y="101" text-anchor="middle" font-size="9.5" font-weight="700" fill="#1e293b">server</text><line x1="442" y1="89" x2="468" y2="89" stroke="#334155" stroke-width="1.7" marker-end="url(#n)"/><text x="455" y="80" text-anchor="middle" font-size="7.5" fill="#475569">bind</text><rect x="470" y="52" width="236" height="120" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="588" y="70" text-anchor="middle" font-size="9" font-weight="700" fill="#14532d">WorkerPool worker · ateom-gvisor</text><rect x="486" y="84" width="204" height="74" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.6"/><text x="588" y="102" text-anchor="middle" font-size="9" font-weight="700" fill="#7c2d12">gVisor sandbox (runsc)</text><text x="588" y="120" text-anchor="middle" font-size="10" font-weight="700" fill="#92400e">ADK actor</text><text x="588" y="138" text-anchor="middle" font-size="8" fill="#b45309">guest kernel ≠ host kernel</text><rect x="300" y="196" width="220" height="40" rx="8" fill="#ede9fe" stroke="#8b5cf6" stroke-width="1.4"/><text x="410" y="214" text-anchor="middle" font-size="9" font-weight="700" fill="#5b21b6">kagent-atelet (DaemonSet)</text><text x="410" y="228" text-anchor="middle" font-size="8" fill="#6d28d9">installs runsc on the node</text><line x1="520" y1="206" x2="588" y2="172" stroke="#8b5cf6" stroke-width="1.7" stroke-dasharray="5 3" marker-end="url(#v)"/><text x="566" y="192" text-anchor="middle" font-size="7.5" fill="#6d28d9">runsc</text><text x="360" y="252" text-anchor="middle" font-size="10" fill="#64748b">A SandboxAgent → ActorTemplate → bound onto a pooled gVisor worker; the agent runs in a runsc sandbox with its own guest kernel.</text></svg></div>

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
# substrate must be enabled first (the 'Enable substrate' cell above).
# Deploy a SandboxAgent: it runs as a gVisor actor on the WorkerPool, not an ordinary pod.
SB=$(mktemp)
cat > "$SB" <<'YAML'
apiVersion: kagent.dev/v1alpha2
kind: SandboxAgent
metadata: { name: substrate-demo, namespace: kagent }
spec:
  type: Declarative
  description: Minimal Go ADK SandboxAgent running on Agent Substrate (gVisor).
  declarative:
    runtime: go                       # go = faster startup; python = full feature set
    modelConfig: default-model-config # auto-created from the anthropic provider
    systemMessage: "You are a helpful assistant running inside a gVisor-sandboxed actor."
  substrate:
    workerPoolRef: { name: kagent-default }
YAML
cat "$SB"
kubectl --context $CTX apply -f "$SB"

# The ate-api-server keeps its OWN worker store and fills it a while AFTER the
# WorkerPool reports its pods Ready. Deploy inside that window and the actor cannot be
# placed ("no free workers available"), and because the controller backs off
# exponentially it does not recover on its own for minutes. Re-applying resets that
# backoff, so retry instead of waiting it out.
for attempt in 1 2 3 4 5 6; do
  if kubectl --context $CTX -n $KAGENT_NS wait sandboxagent/substrate-demo \
       --for=condition=Ready --timeout=45s >/dev/null 2>&1; then
    echo "✓ substrate-demo Ready (attempt $attempt)"
    break
  fi
  MSG=$(kubectl --context $CTX -n $KAGENT_NS get sandboxagent substrate-demo \
          -o jsonpath='{.status.conditions[?(@.type=="Ready")].message}' 2>/dev/null)
  echo "  attempt $attempt: not Ready yet (${MSG:-no status yet}) - resetting the controller backoff"
  kubectl --context $CTX -n $KAGENT_NS delete sandboxagent substrate-demo >/dev/null 2>&1
  kubectl --context $CTX apply -f "$SB" >/dev/null
done
kubectl --context $CTX -n $KAGENT_NS get sandboxagent substrate-demo

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
CYN=$'\e[36m'; GRN=$'\e[32m'; BLD=$'\e[1m'; RST=$'\e[0m'
printf '%s== the pool worker runs the gVisor ateom image ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get pods -l ate.dev/worker-pool=kagent-default \
  -o jsonpath='{range .items[*]}  {.metadata.name}  {.spec.containers[0].image}{"\n"}{end}'
printf '%s== the WorkerPool declares sandboxClass gvisor ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get workerpool kagent-default \
  -o custom-columns='POOL:.metadata.name,CLASS:.spec.sandboxClass,REPLICAS:.spec.replicas'
printf '%s== the SandboxAgent produced a gVisor golden actor ==%s\n' "$GRN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get actortemplate \
  -o custom-columns='ACTORTEMPLATE:.metadata.name,CLASS:.spec.sandboxClass,PHASE:.status.phase'
echo
echo "  Those are declarations. The next cell catches gVisor actually running."

### Catch gVisor in the act

The declarations above say `gvisor`. This proves it. An idle actor is a **snapshot on disk with no
process**, so `runsc` exists only while a turn is being served: fire a request and watch the node
during it. kind runs each Kubernetes node as a container, so we can look straight at the node's
process table.

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
CYN=$'\e[36m'; GRN=$'\e[32m'; BLD=$'\e[1m'; RST=$'\e[0m'
[ -x ./demo-scripts/free-ports.sh ] && ./demo-scripts/free-ports.sh 18083 >/dev/null 2>&1
NODE=$(kubectl --context $CTX -n $KAGENT_NS get pod -l ate.dev/worker-pool=kagent-default \
         -o jsonpath='{.items[0].spec.nodeName}')
echo "worker node (kind runs it as a container): $NODE"

kubectl --context $CTX -n $KAGENT_NS port-forward svc/kagent-controller 18083:8083 >/dev/null 2>&1 &
PF=$!
for i in $(seq 1 15); do curl -s -o /dev/null http://localhost:18083/api/sessions && break; sleep 1; done

# a sandbox actor requires a contextId, which is a kagent session id
SID=$(curl -s --max-time 15 -X POST http://localhost:18083/api/sessions \
        -H 'content-type: application/json' \
        -d '{"agent_ref":"kagent/substrate-demo","name":"gvisor-probe"}' \
      | python3 -c 'import sys,json;print(json.load(sys.stdin).get("data",{}).get("id",""))')
[ -n "$SID" ] || { echo "could not open a session - is substrate-demo Ready?"; kill $PF 2>/dev/null; }
REQ=$(python3 -c "import json,sys;print(json.dumps({'jsonrpc':'2.0','id':'1','method':'message/send','params':{'message':{'role':'user','parts':[{'kind':'text','text':'Write a short paragraph about sandboxes.'}],'messageId':'m1','contextId':sys.argv[1]}}}))" "$SID")

# fire the turn in the background, then watch the node while it is served
curl -s --max-time 90 -X POST "http://localhost:18083/api/a2a-sandboxes/kagent/substrate-demo/" \
     -H 'content-type: application/json' -d "$REQ" >/tmp/sb-gvisor-probe.json 2>&1 &
CURL=$!
CAUGHT=""
for i in $(seq 1 60); do
  CAUGHT=$(docker exec "$NODE" sh -c 'ps -ef | grep "[r]unsc"' 2>/dev/null)
  [ -n "$CAUGHT" ] && break
  sleep 0.3
done
wait $CURL 2>/dev/null

if [ -n "$CAUGHT" ]; then
  printf '%s== gVisor processes serving that turn ==%s\n' "$GRN$BLD" "$RST"
  printf '%s\n' "$CAUGHT" | grep -oE 'runsc-(sandbox|gofer)' | sort | uniq -c | sed 's/^/  /'
  printf '%s== the actor it booted them for ==%s\n' "$CYN$BLD" "$RST"
  printf '%s\n' "$CAUGHT" | grep -oE '/var/lib/ateom-gvisor/actors/[^/]+' | sort -u | sed 's/^/  /'
  printf '%s== the bundles in play ==%s\n' "$CYN$BLD" "$RST"
  printf '%s\n' "$CAUGHT" | grep -oE 'bundles/[a-z]+' | sort -u | sed 's/^/  /'
  echo
  echo "  runsc-sandbox  the gVisor guest kernel, booted for this actor"
  echo "  runsc-gofer    gVisor's file proxy: the actor never touches the host filesystem directly"
  echo "  bundles/pause  the pause/resume bundle, which is the golden-snapshot path (5.4)"
else
  echo "no runsc seen: the turn finished inside the poll window, re-run this cell"
fi
kill $PF 2>/dev/null; wait $PF 2>/dev/null || true

### At rest an actor costs nothing, and every session gets its own

Now that the turn is over, look again. No `runsc`, because the actor was paused back to a snapshot.
The worker keeps one **golden** actor per `SandboxAgent` and one **actor per session**, which is the
isolation boundary that matters for multi-tenancy: two conversations with the same agent are two
separate gVisor sandboxes.

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
CYN=$'\e[36m'; GRN=$'\e[32m'; BLD=$'\e[1m'; RST=$'\e[0m'
NODE=$(kubectl --context $CTX -n $KAGENT_NS get pod -l ate.dev/worker-pool=kagent-default \
         -o jsonpath='{.items[0].spec.nodeName}')
[ -x ./demo-scripts/free-ports.sh ] && ./demo-scripts/free-ports.sh 18083 >/dev/null 2>&1
kubectl --context $CTX -n $KAGENT_NS port-forward svc/kagent-controller 18083:8083 >/dev/null 2>&1 &
PF=$!
for i in $(seq 1 15); do curl -s -o /dev/null http://localhost:18083/api/sessions && break; sleep 1; done

# a SECOND conversation with the SAME agent, so the per-session actors show up side by side
SID=$(curl -s --max-time 15 -X POST http://localhost:18083/api/sessions \
        -H 'content-type: application/json' \
        -d '{"agent_ref":"kagent/substrate-demo","name":"second-conversation"}' \
      | python3 -c 'import sys,json;print(json.load(sys.stdin).get("data",{}).get("id",""))')
if [ -n "$SID" ]; then
  REQ=$(python3 -c "import json,sys;print(json.dumps({'jsonrpc':'2.0','id':'1','method':'message/send','params':{'message':{'role':'user','parts':[{'kind':'text','text':'Say hello in five words.'}],'messageId':'m1','contextId':sys.argv[1]}}}))" "$SID")
  curl -s --max-time 90 -X POST "http://localhost:18083/api/a2a-sandboxes/kagent/substrate-demo/" \
       -H 'content-type: application/json' -d "$REQ" >/dev/null 2>&1
fi
kill $PF 2>/dev/null; wait $PF 2>/dev/null || true

sleep 6   # let the actor pause back down
printf '%s== runsc processes on the node, at rest ==%s\n' "$GRN$BLD" "$RST"
echo "  $(docker exec "$NODE" sh -c 'ps -ef | grep -c "[r]unsc"' 2>/dev/null | tr -d ' \r')"
echo "  Zero: every actor is paused. An idle agent is a snapshot on disk, not a running process."
echo

# Golden snapshots outlive the SandboxAgent that created them, so label each one against
# the live ActorTemplates rather than implying everything on disk is in use.
LIVE=$(kubectl --context $CTX -n $KAGENT_NS get actortemplate \
         -o jsonpath='{range .items[*]}{.status.goldenActorID}{"\n"}{end}' 2>/dev/null)
export LIVE
printf '%s== what the worker stores ==%s\n' "$CYN$BLD" "$RST"
docker exec "$NODE" sh -c 'ls /var/lib/ateom-gvisor/actors/ 2>/dev/null' | while read -r d; do
  case "$d" in
    ate-golden:*)
      if printf '%s\n' "$LIVE" | grep -qx "${d#ate-golden:}"; then
        printf '  %-72s golden snapshot, live SandboxAgent\n' "$d"
      else
        printf '  %-72s golden snapshot, left by a deleted agent\n' "$d"
      fi ;;
    kagent:asr-*) printf '  %-72s actor for ONE session\n' "$d" ;;
    *) printf '  %s\n' "$d" ;;
  esac
done
echo
echo "  One golden snapshot per SandboxAgent, and one gVisor actor per SESSION of it:"
echo "  two conversations with the same agent are two separate sandboxes. Deleting an"
echo "  agent leaves its snapshot behind on the worker, which is why some are marked stale."

## 5.2 · Step 2: warm pool vs cold start

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 238" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="238" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">Step 2 · Warm pool vs cold start</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="v" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#8b5cf6"/></marker></defs><rect x="20" y="58" width="190" height="52" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/><text x="115" y="78" text-anchor="middle" font-size="9" font-weight="700" fill="#14532d">WorkerPool: 2 ready workers</text><text x="115" y="94" text-anchor="middle" font-size="8" fill="#166534">warm, golden resident</text><line x1="210" y1="84" x2="300" y2="84" stroke="#16a34a" stroke-width="1.7" marker-end="url(#g)"/><text x="255" y="76" text-anchor="middle" font-size="7.5" fill="#166534">chat</text><rect x="302" y="64" width="150" height="40" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.4"/><text x="377" y="80" text-anchor="middle" font-size="8.5" font-weight="700" fill="#14532d">bind to warm worker</text><text x="377" y="94" text-anchor="middle" font-size="7.5" fill="#166534">resume from golden</text><rect x="470" y="64" width="90" height="40" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="515" y="88" text-anchor="middle" font-size="13" font-weight="700" fill="#14532d">bind only</text><rect x="20" y="150" width="190" height="52" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.5"/><text x="115" y="170" text-anchor="middle" font-size="9" font-weight="700" fill="#334155">WorkerPool: 0 (scaled down)</text><text x="115" y="186" text-anchor="middle" font-size="8" fill="#475569">cold</text><line x1="210" y1="176" x2="300" y2="176" stroke="#d97706" stroke-width="1.7" marker-end="url(#d)"/><text x="255" y="168" text-anchor="middle" font-size="7.5" fill="#92400e">chat</text><rect x="302" y="156" width="150" height="40" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.4"/><text x="377" y="171" text-anchor="middle" font-size="7.5" font-weight="700" fill="#7c2d12">schedule pod → ateom</text><text x="377" y="184" text-anchor="middle" font-size="7.5" fill="#92400e">start → runsc spawn → bind</text><rect x="470" y="156" width="140" height="40" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.6"/><text x="540" y="180" text-anchor="middle" font-size="13" font-weight="700" fill="#7c2d12">+ provisioning</text><text x="360" y="224" text-anchor="middle" font-size="10" fill="#64748b">Both paths end at a Ready SandboxAgent. The cold one has to provision a worker first, which on a cloud cluster means waiting for a node to join.</text></svg></div>

Both cells below do the **same unit of work**: create a `SandboxAgent` and wait until it is Ready.
The only difference is whether the pool already has a worker to bind onto. That keeps the comparison
honest, because timing an actor bind against a pod schedule would be measuring two different things.

Read the numbers with the platform in mind. On kind the worker image is already cached and there is
no node to provision, so the cold path costs about a second and the two look alike. On a cloud
cluster the cold path is where you wait for a node to join, and that is the wait a warm pool removes.

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
# WARM: capacity already in the pool. The unit of work is deliberately the SAME as the
# cold cell below: create a SandboxAgent, wait until it is Ready.
PROBE=$(mktemp)
cat > "$PROBE" <<'YAML'
apiVersion: kagent.dev/v1alpha2
kind: SandboxAgent
metadata: { name: bind-probe, namespace: kagent }
spec:
  type: Declarative
  description: bind-timing probe
  declarative: { runtime: go, modelConfig: default-model-config, systemMessage: "probe" }
  substrate: { workerPoolRef: { name: kagent-default } }
YAML
cp "$PROBE" /tmp/sb-bind-probe.yaml
kubectl --context $CTX -n $KAGENT_NS delete sandboxagent bind-probe --ignore-not-found >/dev/null 2>&1
kubectl --context $CTX -n $KAGENT_NS wait pod -l ate.dev/worker-pool=kagent-default \
  --for=condition=Ready --timeout=180s >/dev/null
echo "warm workers ready: $(kubectl --context $CTX -n $KAGENT_NS get pod -l ate.dev/worker-pool=kagent-default --no-headers | grep -c Running)"

T0=$(python3 -c 'import time;print(time.time())')
kubectl --context $CTX apply -f "$PROBE" >/dev/null
kubectl --context $CTX -n $KAGENT_NS wait sandboxagent/bind-probe --for=condition=Ready --timeout=180s >/dev/null
T1=$(python3 -c 'import time;print(time.time())')
python3 -c "print(f'WARM: actor Ready in {$T1-$T0:.2f}s (no pod scheduled)')" | tee /tmp/sb-warm.txt
kubectl --context $CTX -n $KAGENT_NS delete sandboxagent bind-probe >/dev/null 2>&1

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
# COLD: no capacity at all. Same end state (a Ready SandboxAgent), but the pool has to
# provision a worker first. Wait on the worker pods being DELETED, not on
# status.replicas=0: 'wait --for=jsonpath={.status.replicas}=0' races the status update.
kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":0}}' >/dev/null
kubectl --context $CTX -n $KAGENT_NS wait pod -l ate.dev/worker-pool=kagent-default --for=delete --timeout=180s >/dev/null
echo "pool cold (0 workers)"

T0=$(python3 -c 'import time;print(time.time())')
kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":1}}' >/dev/null
until kubectl --context $CTX -n $KAGENT_NS get pod -l ate.dev/worker-pool=kagent-default -o name 2>/dev/null | grep -q pod/; do sleep 1; done
kubectl --context $CTX -n $KAGENT_NS wait pod -l ate.dev/worker-pool=kagent-default --for=condition=Ready --timeout=300s >/dev/null
T1=$(python3 -c 'import time;print(time.time())')

# then the same bind as the warm cell, onto that fresh worker
kubectl --context $CTX apply -f /tmp/sb-bind-probe.yaml >/dev/null
for attempt in 1 2 3 4; do
  kubectl --context $CTX -n $KAGENT_NS wait sandboxagent/bind-probe --for=condition=Ready --timeout=45s >/dev/null 2>&1 && break
  kubectl --context $CTX -n $KAGENT_NS delete sandboxagent bind-probe >/dev/null 2>&1
  kubectl --context $CTX apply -f /tmp/sb-bind-probe.yaml >/dev/null
done
T2=$(python3 -c 'import time;print(time.time())')

echo
cat /tmp/sb-warm.txt 2>/dev/null
python3 -c "print(f'COLD: worker provisioned in {$T1-$T0:.2f}s, then actor Ready in {$T2-$T1:.2f}s = {$T2-$T0:.2f}s total')"
echo
echo "  Read this honestly: on kind the worker image is already cached and there is no"
echo "  node to provision, so the cold worker comes back in about a second and the two"
echo "  paths look alike. On a cloud cluster the cold path is where you wait for a node"
echo "  to join, which is the wait the warm pool removes."

kubectl --context $CTX -n $KAGENT_NS delete sandboxagent bind-probe >/dev/null 2>&1
kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":2}}' >/dev/null
echo "  pool restored to 2 workers"

## 5.3 · Compare actor and pod resource use

Two properties matter here: standing up another agent does **not** cost another pod, and the
workload really is sandboxed. Below, three more `SandboxAgent`s bind onto the workers already
running, timed to the millisecond, and then the **same three agents as ordinary `Agent`s** for
comparison, which is where the pods and the reserved memory show up. Many isolated actors, few pods.

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
CYN=$'\e[36m'; GRN=$'\e[32m'; BLD=$'\e[1m'; RST=$'\e[0m'
kubectl --context $CTX -n $KAGENT_NS wait pod -l ate.dev/worker-pool=kagent-default \
  --for=condition=Ready --timeout=180s >/dev/null
before=$(kubectl --context $CTX -n $KAGENT_NS get pod -l ate.dev/worker-pool=kagent-default --no-headers | grep -c Running)
printf '%s== worker pods before: %s ==%s\n' "$CYN$BLD" "$before" "$RST"

printf '%s== 3 more sandboxed actors, timed to the millisecond ==%s\n' "$CYN$BLD" "$RST"
for n in 2 3 4; do
  cat > /tmp/sb-density-$n.yaml <<YAML
apiVersion: kagent.dev/v1alpha2
kind: SandboxAgent
metadata: { name: substrate-demo-$n, namespace: kagent }
spec:
  type: Declarative
  description: density demo actor $n
  declarative: { runtime: go, modelConfig: default-model-config, systemMessage: "actor $n" }
  substrate: { workerPoolRef: { name: kagent-default } }
YAML
  T0=$(python3 -c 'import time;print(time.time())')
  kubectl --context $CTX apply -f /tmp/sb-density-$n.yaml >/dev/null
  for attempt in 1 2 3 4; do
    kubectl --context $CTX -n $KAGENT_NS wait sandboxagent/substrate-demo-$n --for=condition=Ready --timeout=45s >/dev/null 2>&1 && break
    kubectl --context $CTX -n $KAGENT_NS delete sandboxagent substrate-demo-$n >/dev/null 2>&1
    kubectl --context $CTX apply -f /tmp/sb-density-$n.yaml >/dev/null
  done
  T1=$(python3 -c 'import time;print(time.time())')
  python3 -c "print(f'   substrate-demo-$n Ready in {($T1-$T0)*1000:.0f} ms')"
done
after=$(kubectl --context $CTX -n $KAGENT_NS get pod -l ate.dev/worker-pool=kagent-default --no-headers | grep -c Running)
actors=$(kubectl --context $CTX -n $KAGENT_NS get actortemplate --no-headers | wc -l | tr -d ' ')
printf '%s   %s gVisor actors, worker pods still %s (was %s)%s\n' "$GRN$BLD" "$actors" "$after" "$before" "$RST"

# ── the baseline this is actually being compared against ─────────────────────
printf '\n%s== now the alternative: 3 ORDINARY agents, each in its own pod ==%s\n' "$CYN$BLD" "$RST"
for n in 1 2 3; do
  kubectl --context $CTX apply -f - >/dev/null <<YAML
apiVersion: kagent.dev/v1alpha2
kind: Agent
metadata: { name: pod-baseline-$n, namespace: kagent }
spec:
  type: Declarative
  description: ordinary pod-backed agent $n
  declarative: { modelConfig: default-model-config, systemMessage: "baseline $n" }
YAML
done
for n in 1 2 3; do
  kubectl --context $CTX -n $KAGENT_NS wait agent/pod-baseline-$n --for=condition=Ready --timeout=180s >/dev/null 2>&1
done
kubectl --context $CTX -n $KAGENT_NS get deploy -l app.kubernetes.io/managed-by=kagent 2>/dev/null | grep pod-baseline
MEM=$(kubectl --context $CTX -n $KAGENT_NS get pod -l kagent=pod-baseline-1 \
        -o jsonpath='{.items[0].spec.containers[0].resources.requests.memory}' 2>/dev/null)
LIM=$(kubectl --context $CTX -n $KAGENT_NS get pod -l kagent=pod-baseline-1 \
        -o jsonpath='{.items[0].spec.containers[0].resources.limits.memory}' 2>/dev/null)
bpods=$(kubectl --context $CTX -n $KAGENT_NS get pod -l app.kubernetes.io/managed-by=kagent --no-headers 2>/dev/null | grep -c pod-baseline)
echo
printf '%s== the comparison ==%s\n' "$GRN$BLD" "$RST"
printf '   3 ordinary Agents   -> %s pods, each requesting %s (limit %s)\n' "$bpods" "${MEM:-?}" "${LIM:-?}"
printf '   3 SandboxAgents     -> 0 extra pods, packed onto the %s workers already running\n' "$after"
echo "   Same three agents. One costs pods and reserved memory per agent; the other costs"
echo "   a snapshot each on a fleet you sized once."

echo
echo "clean up the extras (leave substrate-demo):"
kubectl --context $CTX -n $KAGENT_NS delete sandboxagent substrate-demo-2 substrate-demo-3 substrate-demo-4 --ignore-not-found
kubectl --context $CTX -n $KAGENT_NS delete agent pod-baseline-1 pod-baseline-2 pod-baseline-3 --ignore-not-found

## 5.4 · golden actor + snapshot resume

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 232" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="232" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">Golden actor + snapshot resume</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker><marker id="v" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#8b5cf6"/></marker></defs><rect x="280" y="64" width="160" height="54" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="360" y="86" text-anchor="middle" font-size="11" font-weight="700" fill="#14532d">golden actor</text><text x="360" y="102" text-anchor="middle" font-size="8" fill="#166534">memory snapshot on pause</text><rect x="500" y="66" width="200" height="50" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="600" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#334155">gs:// snapshot bucket</text><text x="600" y="101" text-anchor="middle" font-size="8" fill="#475569">cross-worker persistence</text><line x1="440" y1="86" x2="498" y2="86" stroke="#334155" stroke-width="1.7" stroke-dasharray="5 3" marker-end="url(#n)"/><text x="469" y="78" text-anchor="middle" font-size="7.5" fill="#475569">snapshot</text><rect x="160" y="150" width="240" height="50" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/><text x="280" y="170" text-anchor="middle" font-size="10" font-weight="700" fill="#1e293b">new actor bind</text><text x="280" y="186" text-anchor="middle" font-size="8.5" fill="#475569">ResumeGoldenActor</text><line x1="360" y1="118" x2="300" y2="148" stroke="#16a34a" stroke-width="1.7" marker-end="url(#g)"/><text x="345" y="138" text-anchor="middle" font-size="7.5" fill="#166534">resume</text><rect x="430" y="150" width="270" height="50" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.4"/><text x="565" y="169" text-anchor="middle" font-size="9" font-weight="700" fill="#7c2d12">internal engine reconcile</text><text x="565" y="185" text-anchor="middle" font-size="8" fill="#92400e">no operator verb · automatic</text><text x="360" y="222" text-anchor="middle" font-size="10" fill="#64748b">Idle actor snapshots its memory, then resumes from a golden snapshot, not a cold boot. Automatic, no kubectl verb.</text></svg></div>

Pause, snapshot and resume are **automatic**: there is no kubectl verb, `ResumeGoldenActor` is an
internal phase of the substrate engine. The cell below shows the golden actor behind each
`SandboxAgent` and the snapshot it keeps on the worker. That snapshot is what a bind resumes from
instead of cold-booting a runtime, which is why the binds in 5.3 land in a few hundred milliseconds.

Resuming from a golden works with no configuration, and neither does persisting it. kagent already
points substrate at object storage and stamps every `ActorTemplate` with its own
`snapshotsConfig.location`, so snapshots survive a worker and an actor can come back somewhere else.
On this cluster the backend is the in-cluster RustFS bucket (`atelet` runs with an S3 backend, so the
`gs://` scheme in the location is nominal). To use your own bucket, set the location yourself:

```yaml
apiVersion: kagent.dev/v1alpha2
kind: SandboxAgent
metadata: { name: substrate-demo, namespace: kagent }
spec:
  substrate:
    workerPoolRef: { name: kagent-default }
    snapshotsConfig:
      location: gs://<your-bucket>/kagent/substrate-demo/
```

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
CYN=$'\e[36m'; GRN=$'\e[32m'; BLD=$'\e[1m'; RST=$'\e[0m'
NODE=$(kubectl --context $CTX -n $KAGENT_NS get pod -l ate.dev/worker-pool=kagent-default \
         -o jsonpath='{.items[0].spec.nodeName}')
printf '%s== every SandboxAgent has a golden actor, keyed by UUID ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get actortemplate \
  -o custom-columns='ACTORTEMPLATE:.metadata.name,CLASS:.spec.sandboxClass,PHASE:.status.phase,GOLDEN:.status.goldenActorID'

printf '\n%s== nobody configured this: kagent gave each one object storage ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get actortemplate \
  -o jsonpath='{range .items[*]}  {.metadata.name}{"  ->  "}{.spec.snapshotsConfig.location}{"\n"}{end}'
echo "  atelet writes them with an S3 backend:"
kubectl --context $CTX -n $KAGENT_NS get ds kagent-atelet \
  -o jsonpath='{range .spec.template.spec.containers[0].env[*]}{.name}{"="}{.value}{"\n"}{end}' \
  | grep -E 'ATE_STORAGE_BACKEND|AWS_ENDPOINT_URL' | sed 's/^/    /'

printf '\n%s== and the golden snapshot is also resident on this worker ==%s\n' "$GRN$BLD" "$RST"
for id in $(kubectl --context $CTX -n $KAGENT_NS get actortemplate \
              -o jsonpath='{range .items[*]}{.status.goldenActorID}{"\n"}{end}' 2>/dev/null); do
  if docker exec "$NODE" sh -c "test -d '/var/lib/ateom-gvisor/actors/ate-golden:$id'" 2>/dev/null; then
    echo "  ate-golden:$id  present"
  else
    echo "  ate-golden:$id  not on this worker (held by another, or only in object storage)"
  fi
done
echo
echo "  The UUIDs line up: a golden snapshot is what a bind resumes from, which is how"
echo "  5.3 gets a Ready actor in a few hundred milliseconds instead of cold-booting a"
echo "  runtime. It is resident on the worker for speed AND has a URI in object storage,"
echo "  which is what lets an actor come back on a different worker."

## 5.5 · The worker fleet

A `WorkerPool` is a fleet of pre-warmed worker pods, and every agent runs as a gVisor actor packed onto one of them. Adding an agent binds a new actor onto a warm worker in about a second; it does not cost a pod. The fleet scales on its own, so you add workers for capacity without touching the agents.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 300" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="760" height="300" rx="10" fill="#f8fafc"/><text x="380" y="26" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Agent Substrate: one WorkerPool, many gVisor actors</text><text x="380" y="44" text-anchor="middle" font-size="10.5" fill="#475569">each SandboxAgent runs as a gVisor actor; actors pack onto shared, pre-warmed worker pods</text><rect x="20" y="58" width="720" height="192" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="1.6"/><text x="36" y="78" font-size="11" font-weight="700" fill="#1e40af">WorkerPool · kagent-default · sandboxClass gvisor</text><rect x="36" y="90" width="216" height="148" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="144" y="110" text-anchor="middle" font-size="11" font-weight="700" fill="#334155">worker pod 1</text><rect x="54" y="120" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="144" y="139" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · checkout</text><text x="144" y="154" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="54" y="176" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="144" y="195" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · search</text><text x="144" y="210" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="272" y="90" width="216" height="148" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="380" y="110" text-anchor="middle" font-size="11" font-weight="700" fill="#334155">worker pod 2</text><rect x="290" y="120" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="380" y="139" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · pricing</text><text x="380" y="154" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="290" y="176" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="380" y="195" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · support</text><text x="380" y="210" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="508" y="90" width="216" height="148" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="616" y="110" text-anchor="middle" font-size="11" font-weight="700" fill="#334155">worker pod 3</text><rect x="526" y="120" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="616" y="139" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · summarizer</text><text x="616" y="154" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><rect x="526" y="176" width="180" height="46" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.3"/><text x="616" y="195" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">actor · planner</text><text x="616" y="210" text-anchor="middle" font-size="8" fill="#166534">gVisor (runsc) sandbox</text><text x="380" y="278" text-anchor="middle" font-size="10.5" fill="#334155">Add an agent and a new actor binds onto a warm worker in about a second, not a new pod. Scale the pool to add workers.</text></svg></div>

Below: the live fleet and an elastic resize (2 to 4 workers and back, with the actors untouched).

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
CYN=$'\e[36m'; GRN=$'\e[32m'; BLD=$'\e[1m'; RST=$'\e[0m'
printf '%s== the worker fleet, each pod hosts gVisor actors ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS get pods -l ate.dev/worker-pool=kagent-default -o wide

printf '%s== the fleet is elastic: scale 2 -> 4 workers, actors untouched ==%s\n' "$CYN$BLD" "$RST"
kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":4}}' >/dev/null
n=0
for i in $(seq 1 30); do
  n=$(kubectl --context $CTX -n $KAGENT_NS get pods -l ate.dev/worker-pool=kagent-default --field-selector=status.phase=Running --no-headers 2>/dev/null | wc -l | tr -d ' ')
  [ "$n" -ge 4 ] && break
  sleep 2
done
actors=$(kubectl --context $CTX -n $KAGENT_NS get actortemplates --no-headers | wc -l | tr -d ' ')
echo "  fleet grew to $n worker pods; gVisor actors still: $actors (unaffected)"
kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":2}}' >/dev/null
echo "  scaled back to 2, workers are cattle, the actors carried on"

## 5.6 · Chat with the sandboxed agent

The proof that matters: talk to it. A SandboxAgent is an A2A server. Regular agents answer on `/api/a2a/<ns>/<name>/`; sandboxed ones answer on `/api/a2a-sandboxes/<ns>/<name>/`. Every message carries a `contextId`, which is a kagent session id, so we open a session and then send the prompt. The reply comes straight from the ADK agent running inside the gVisor actor.

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
CYN=$'\e[36m'; GRN=$'\e[32m'; BLD=$'\e[1m'; RST=$'\e[0m'
kubectl --context $CTX -n $KAGENT_NS port-forward svc/kagent-controller 18083:8083 >/dev/null 2>&1 &
PF=$!
for i in $(seq 1 10); do curl -s -o /dev/null http://localhost:18083/api/sessions 2>/dev/null && break; sleep 1; done

ask() {
  # 1) open a session; its id is the A2A contextId a sandbox actor requires
  local sid=$(curl -s --max-time 15 -X POST http://localhost:18083/api/sessions \
      -H 'content-type: application/json' -d '{"agent_ref":"kagent/substrate-demo","name":"chat"}' \
      | python3 -c 'import sys,json; print(json.load(sys.stdin).get("data",{}).get("id",""))' 2>/dev/null)
  # no session means the agent is not Ready (or the port-forward died): say so plainly
  if [ -z "$sid" ]; then
    printf '  cannot open a session for kagent/substrate-demo - check it is Ready:\n'
    kubectl --context $CTX -n $KAGENT_NS get sandboxagent substrate-demo 2>&1 | sed 's/^/    /'
    return 1
  fi
  # 2) send the prompt over A2A (message/send) to the gVisor-sandboxed actor
  local req=$(python3 -c "import json,sys; print(json.dumps({'jsonrpc':'2.0','id':'1','method':'message/send','params':{'message':{'role':'user','parts':[{'kind':'text','text':sys.argv[1]}],'messageId':'m1','contextId':sys.argv[2]}}}))" "$1" "$sid")
  printf '%s> %s%s\n' "$CYN$BLD" "$1" "$RST"
  curl -s --max-time 90 -X POST http://localhost:18083/api/a2a-sandboxes/kagent/substrate-demo/ \
      -H 'content-type: application/json' -d "$req" \
    | python3 -c '
import sys,json
d=json.load(sys.stdin); r=d.get("result",{})
arts=r.get("artifacts",[])
if arts:
    print("  "+arts[0]["parts"][0]["text"])
    app=next((m.get("metadata",{}).get("adk_app_name") for m in r.get("history",[]) if m.get("metadata",{}).get("adk_app_name")),None)
    if app: print("     (answered by "+app+", running as a gVisor actor)")
else:
    print("  (no answer: "+str(d.get("error",{}).get("message","?"))+")")'
}

ask "In one sentence, what is 17 times 3?"
ask "Name one benefit of running an agent in a gVisor sandbox. One sentence."

kill $PF 2>/dev/null; wait $PF 2>/dev/null || true

## Tear down

In [ ]:
: "${CTX:=kind-substrate}" "${KAGENT_NS:=kagent}"
# every object any cell in this notebook can create
kubectl --context $CTX -n $KAGENT_NS delete sandboxagent \
  substrate-demo substrate-demo-2 substrate-demo-3 substrate-demo-4 bind-probe --ignore-not-found
kubectl --context $CTX -n $KAGENT_NS delete agent \
  pod-baseline-1 pod-baseline-2 pod-baseline-3 --ignore-not-found
kubectl --context $CTX -n $KAGENT_NS patch workerpool kagent-default --type=merge -p '{"spec":{"replicas":2}}' >/dev/null 2>&1
rm -f /tmp/sb-bind-probe.yaml /tmp/sb-density-*.yaml /tmp/sb-warm.txt /tmp/sb-gvisor-probe.json
echo "✓ demo objects removed; the pool is back to 2 workers"
# to fully remove substrate (and hand kagent back to Part 4):
#   helm --kube-context $CTX upgrade kagent "$KENT_CHART" -n $KAGENT_NS --reuse-values \
#     --set substrate.enabled=false --set substrateWorkerPool.create=false --set controller.substrate.enabled=false
#   kubectl --context $CTX -n $KAGENT_NS delete workerpool kagent-default --ignore-not-found